In [12]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model


In [13]:
BASE_PATH = r"E:\Final Year Project\dataset\URTIC"

train_cold = os.path.join(BASE_PATH, "train", "cold")
train_noncold = os.path.join(BASE_PATH, "train", "noncold")

dev_cold = os.path.join(BASE_PATH, "develop", "cold")
dev_noncold = os.path.join(BASE_PATH, "develop", "noncold")

test_cold = os.path.join(BASE_PATH, "test", "cold")
test_noncold = os.path.join(BASE_PATH, "test", "noncold")


In [ ]:
MAX_FRAMES = 300

def extract_mfcc_sequence(file_path, sr=16000): 
    try:
        signal, sr = librosa.load(file_path, sr=sr)

        if len(signal) < sr * 0.3:
            return None

        alpha = 0.97
        emphasized = np.append(signal[0], signal[1:] - alpha * signal[:-1])

        emphasized, _ = librosa.effects.trim(emphasized, top_db=20)

        frame_length = int(0.02 * sr)
        hop_length = int(0.01 * sr)

        mfcc = librosa.feature.mfcc(
            y=emphasized,
            sr=sr,
            n_mfcc=13,
            n_fft=512,
            win_length=frame_length,
            hop_length=hop_length,
            window="hamming"
        )

        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)

        mfcc_39 = np.vstack([mfcc, delta, delta2]).T

        if mfcc_39.shape[0] < MAX_FRAMES:
            pad = MAX_FRAMES - mfcc_39.shape[0]
            mfcc_39 = np.pad(mfcc_39, ((0, pad), (0, 0)))
        else:
            mfcc_39 = mfcc_39[:MAX_FRAMES]

        return mfcc_39.astype(np.float32)

    except:
        return None


In [15]:
def load_dataset_full(cold_dir, noncold_dir):

    X, y = [], []

    cold_files = [f for f in os.listdir(cold_dir) if f.endswith(".wav")]
    noncold_files = [f for f in os.listdir(noncold_dir) if f.endswith(".wav")]

    np.random.shuffle(cold_files)
    np.random.shuffle(noncold_files)

    print("Cold files:", len(cold_files))
    print("Non-cold files:", len(noncold_files))

    for f in cold_files:
        feats = extract_mfcc_sequence(os.path.join(cold_dir, f))
        if feats is not None:
            X.append(feats)
            y.append(1)

    for f in noncold_files:
        feats = extract_mfcc_sequence(os.path.join(noncold_dir, f))
        if feats is not None:
            X.append(feats)
            y.append(0)

    X = np.array(X)
    y = np.array(y)

    idx = np.random.permutation(len(X))
    return X[idx], y[idx]


In [16]:
X_train, y_train = load_dataset_full(train_cold, train_noncold)
X_dev, y_dev = load_dataset_full(dev_cold, dev_noncold)
X_test, y_test = load_dataset_full(test_cold, test_noncold)

print("Train:", X_train.shape)
print("Dev:", X_dev.shape)
print("Test:", X_test.shape)


Cold files: 970
Non-cold files: 8532
Cold files: 1011
Non-cold files: 8585


C:\Users\user\AppData\Local\Temp\ipykernel_6916\3024600717.py:5: UserWarning: PySoundFile failed. Trying audioread instead.
  signal, sr = librosa.load(file_path, sr=sr)
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Cold files: 894
Non-cold files: 8655
Train: (9502, 300, 39)
Dev: (9595, 300, 39)
Test: (9549, 300, 39)


In [17]:
N, T, F = X_train.shape
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, F)).reshape(N, T, F)

N, T, F = X_dev.shape
X_dev = scaler.transform(X_dev.reshape(-1, F)).reshape(N, T, F)

N, T, F = X_test.shape
X_test = scaler.transform(X_test.reshape(-1, F)).reshape(N, T, F)


In [1]:
def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        eps = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, eps, 1 - eps)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        return tf.reduce_mean(-alpha * tf.pow(1 - p_t, gamma) * tf.math.log(p_t))
    return loss


In [19]:
def transformer_encoder(x, head_size, num_heads, ff_dim, dropout=0.4):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(x, x)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dropout(dropout)(ff)
    ff = Dense(ff_dim)(ff)

    return LayerNormalization(epsilon=1e-6)(x + ff)


In [20]:
inputs = Input(shape=(MAX_FRAMES, 39))

x = Dense(256)(inputs)
x = transformer_encoder(x, 64, 8, 256)
x = transformer_encoder(x, 64, 8, 256)

x = GlobalAveragePooling1D()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.4)(x)

outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=focal_loss(),
    metrics=["accuracy"]
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 300, 39)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 300, 256)  │     10,240 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 300, 256)  │    526,080 │ dense_7[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 300, 256)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 300, 256)  │          0 │ dense_7[0][0],    │
│                     │                   │            │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 300, 256)  │        512 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 300, 256)  │     65,792 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 300, 256)  │          0 │ dense_8[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 300, 256)  │     65,792 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 300, 256)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 300, 256)  │        512 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 300, 256)  │    526,080 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 300, 256)  │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 300, 256)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 300, 256)  │        512 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 300, 256)  │     65,792 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 300, 256)  │          0 │ dense_10[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 300, 256)  │     65,792 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 300, 256)  │          0 │ layer_normalizat

 Total params: 1,360,641 (5.19 MB)

 Trainable params: 1,360,641 (5.19 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_dev, y_dev),
    epochs=100,
    batch_size=32,
    callbacks=[early],
    shuffle=True
)


Epoch 1/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 447s 1s/step - accuracy: 0.8883 - loss: 0.0224 - val_accuracy: 0.8987 - val_loss: 0.0257
Epoch 2/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 414s 1s/step - accuracy: 0.9015 - loss: 0.0187 - val_accuracy: 0.8969 - val_loss: 0.0265
Epoch 3/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 425s 1s/step - accuracy: 0.9063 - loss: 0.0173 - val_accuracy: 0.8859 - val_loss: 0.0280
Epoch 4/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 410s 1s/step - accuracy: 0.9168 - loss: 0.0154 - val_accuracy: 0.8811 - val_loss: 0.0348
Epoch 5/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 398s 1s/step - accuracy: 0.9269 - loss: 0.0131 - val_accuracy: 0.8681 - val_loss: 0.0384
Epoch 6/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 409s 1s/step - accuracy: 0.9392 - loss: 0.0109 - val_accuracy: 0.8584 - val_loss: 0.0478
Epoch 7/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 391s 1s/step - accuracy: 0.9490 - loss: 0.0091 - val_accuracy: 0.8828 - val_loss: 0.0571
Epoch 8/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 385s 1s/step - accuracy: 0.9574 - loss: 0.0073 - 

In [22]:
y_dev_prob = model.predict(X_dev).ravel()

fpr, tpr, thresholds = roc_curve(y_dev, y_dev_prob)
roc_auc = auc(fpr, tpr)

youden = tpr - fpr
best_idx = np.argmax(youden)
best_thresh = thresholds[best_idx]

print("DEV AUC:", roc_auc)
print("Best threshold:", best_thresh)


300/300 ━━━━━━━━━━━━━━━━━━━━ 88s 292ms/step
DEV AUC: 0.6456670012896351
Best threshold: 0.3241279721260071


In [27]:
y_test_prob = model.predict(X_test).ravel()
y_test_pred = (y_test_prob >= 0.25).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=["Non-Cold", "Cold"]))


299/299 ━━━━━━━━━━━━━━━━━━━━ 95s 317ms/step
Confusion Matrix:
[[5564 3091]
 [ 320  574]]

Classification Report:
              precision    recall  f1-score   support

    Non-Cold       0.95      0.64      0.77      8655
        Cold       0.16      0.64      0.25       894

    accuracy                           0.64      9549
   macro avg       0.55      0.64      0.51      9549
weighted avg       0.87      0.64      0.72      9549



In [24]:
cm = confusion_matrix(y_test, y_test_pred)
TN, FP, FN, TP = cm.ravel()

recall_cold = TP / (TP + FN)
recall_noncold = TN / (TN + FP)
UAR = (recall_cold + recall_noncold) / 2

print("Cold Recall:", recall_cold)
print("Non-Cold Recall:", recall_noncold)
print("UAR:", UAR)


Cold Recall: 0.33109619686800895
Non-Cold Recall: 0.8848064702484113
UAR: 0.6079513335582101
